In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from tensorflow.keras import layers,models,utils
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical


In [2]:
import string

# All uppercase letters except 'J' and 'Z'
classes = [ch for ch in string.ascii_uppercase if ch not in ['J', 'Z']]
label_map = {ch: idx for idx, ch in enumerate(classes)}  # {'A': 0, 'B': 1, ..., 'Y': 23}


In [ ]:
import cv2
import os
import numpy as np

def load_images_from_folder(folder_path, label_map):
    data = []
    labels = []

    for label in os.listdir(folder_path):
        if label not in label_map:
            continue
        label_folder = os.path.join(folder_path, label)
        for file in os.listdir(label_folder):
            img_path = os.path.join(label_folder, file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img_resized = cv2.resize(img, (28, 28))  # match MNIST size
            data.append(img_resized)
            labels.append(label_map[label])
    
    return np.array(data), np.array(labels)

# Load both good and bad lighting sets
X_good, y_good = load_images_from_folder("C:\\Users\\tanin\\OneDrive\\Desktop\\emerging_trends\\Sign Language\\clean_good_lighting", label_map)
X_bad, y_bad = load_images_from_folder("C:\\Users\\tanin\\OneDrive\\Desktop\\emerging_trends\\Sign Language\\clean_bad_lighting", label_map)
X_extra, y_extra =load_images_from_folder("C:\\Users\\tanin\\OneDrive\\Desktop\\emerging_trends\\Sign Language\\clean_extra", label_map)

# Combine your custom image dataset
X_custom = np.concatenate([X_good, X_bad,X_extra])
y_custom = np.concatenate([y_good, y_bad,y_extra])


In [13]:
def load_sign_mnist_csv(path, label_map):
    df = pd.read_csv(path)
    
    # Map label integers to letters, then to new indices
    original_labels = {i: ch for i, ch in enumerate(string.ascii_uppercase)}
    df['label'] = df['label'].map(original_labels)
    df = df[df['label'].isin(label_map)]
    df['label'] = df['label'].map(label_map)

    X = df.drop('label', axis=1).values.reshape(-1, 28, 28).astype('uint8')
    y = df['label'].values
    return X, y

X_train_mnist, y_train_mnist = load_sign_mnist_csv('sign_mnist_train.csv', label_map)
X_test_mnist, y_test_mnist = load_sign_mnist_csv('sign_mnist_test.csv', label_map)

X_mnist = np.concatenate([X_train_mnist, X_test_mnist])
y_mnist = np.concatenate([y_train_mnist, y_test_mnist])

In [ ]:
# Combine MNIST and custom
X_all = np.concatenate([X_mnist, X_custom])
y_all = np.concatenate([y_mnist, y_custom])

# Normalize and reshape
X_all = X_all / 255.0
X_all = X_all.reshape(-1, 28, 28, 1)


In [21]:
X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

num_classes = 24
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes)

In [22]:
model = models.Sequential()

# Convolutional layer block 1
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Dropout(0.4))

# Convolutional layer block 2
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Dropout(0.4))

# Convolutional layer block 3
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Dropout(0.4))

# Flatten the feature maps to a 1D vector
model.add(layers.Flatten())

# Fully connected (dense) layer
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.8))

# Output layer (24 classes: A to Y)
model.add(layers.Dense(24, activation='softmax'))

C:\Users\tanin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [23]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display model summary
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 3, 3, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 1, 1, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 1, 1, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 112,280 (438.59 KB)

 Trainable params: 112,280 (438.59 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
# Fit the model to your data
history = model.fit(X_train, y_train_cat, epochs=15, batch_size=32, validation_data=(X_val, y_val_cat))


Epoch 1/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.0416 - loss: 3.1823 - val_accuracy: 0.1570 - val_loss: 2.8205
Epoch 2/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.1723 - loss: 2.6486 - val_accuracy: 0.6366 - val_loss: 1.2291
Epoch 3/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.4461 - loss: 1.6309 - val_accuracy: 0.8237 - val_loss: 0.7342
Epoch 4/15
377/938 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.5547 - loss: 1.2925

KeyboardInterrupt: 

In [42]:
# Evaluate the model on the validation set
val_loss, val_accuracy = model.evaluate(X_val, y_val_cat)
print(f'Validation Accuracy: {val_accuracy * 100:.2f}%')


250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9995 - loss: 0.0020
Validation Accuracy: 99.95%


In [44]:
# Save the model for later use
model.save('asl_sign_language_model.keras')
